# Pulizia dei dataset Personaggi FiveThirtyEight
## Operazioni eseguite sul dataset Marvel

1. **Cancellazione della colonna `GSM`** perché presenta una quasi totale mancanza di dati.
2. **Cancellazione della colonna `FIRST APPEARANCE`** perché contiene la stessa informazione temporale di `Year` ed è quindi ridondante.
3. **Rinomina della colonna `Year`** in `Year First Appearance`, per rendere il significato della variabile più esplicito.
4. **Gestione dei valori vuoti nelle colonne categoriche**: i valori mancanti di `ID`, `ALIGN`, `EYE`, `HAIR` e `SEX` vengono rappresentati con `Unknown`. Questa scelta non assegna un valore inventato: indica semplicemente che l'informazione non è disponibile nel dataset.
5. **Mantenimento dei valori mancanti numerici**: i valori mancanti di `APPEARANCES` e `Year First Appearance` restano `NaN`, perché non possono essere sostituiti senza introdurre informazioni non verificate.

In [1]:
from pathlib import Path

import pandas as pd

BASE_PATH = Path('/Users/marcogreco/ComicGraph')
DATASET_PATH = BASE_PATH / 'Dataset' / 'Dataset_Originali'

MARVEL_CHARACTERS_PATH = DATASET_PATH / 'PersonaggiMarvelFive.csv'
DC_CHARACTERS_PATH = DATASET_PATH / 'PersonaggiDCFive.csv'

MARVEL_COMICS_PATH = DATASET_PATH / 'Marvel_Comics.csv'
DC_COMICS_PATH = DATASET_PATH / 'DC_Comics.csv'

In [2]:
# Caricamento del dataset Marvel 
marvel_characters = pd.read_csv(MARVEL_CHARACTERS_PATH)

marvel_characters = marvel_characters.drop(
    columns=['GSM', 'FIRST APPEARANCE']
)
marvel_characters = marvel_characters.rename(
    columns={'Year': 'Year First Appearance'}
)

print('Dimensioni:', marvel_characters.shape)
print('Colonne:', list(marvel_characters.columns))

missing_marvel = (
    marvel_characters.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame('valori_mancanti')
)
missing_marvel['percentuale'] = (
    missing_marvel['valori_mancanti'] / len(marvel_characters) * 100
).round(2)

missing_marvel[missing_marvel['valori_mancanti'] > 0]

Dimensioni: (16376, 11)
Colonne: ['page_id', 'name', 'urlslug', 'ID', 'ALIGN', 'EYE', 'HAIR', 'SEX', 'ALIVE', 'APPEARANCES', 'Year First Appearance']


,valori_mancanti,percentuale
EYE,9767,59.64
HAIR,4264,26.04
ID,3770,23.02
ALIGN,2812,17.17
APPEARANCES,1096,6.69
SEX,854,5.21
Year First Appearance,815,4.98
ALIVE,3,0.02


In [3]:
# Gestione dei valori mancanti
categorical_columns = ['ID', 'ALIGN', 'EYE', 'HAIR', 'SEX']

marvel_characters[categorical_columns] = (
    marvel_characters[categorical_columns].fillna('Unknown')
)

removed_alive_rows = marvel_characters[marvel_characters['ALIVE'].isna()].copy()
marvel_characters = marvel_characters.dropna(subset=['ALIVE']).copy()

print('Record eliminati per ALIVE mancante:')
print(removed_alive_rows[['page_id', 'name']].to_string(index=False))
print('\nNuove dimensioni:', marvel_characters.shape)

remaining_missing = (
    marvel_characters.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame('valori_mancanti')
)
remaining_missing[remaining_missing['valori_mancanti'] > 0]

Record eliminati per ALIVE mancante:
 page_id        name
  541449      Mj7711
  714409 Sharjeel786
  462671   TOR\/test

Nuove dimensioni: (16373, 11)


,valori_mancanti
APPEARANCES,1093
Year First Appearance,812


In [4]:
# Salvataggio del dataset Marvel modificato
OUTPUT_PATH = BASE_PATH / 'Dataset' / 'Puliti'
OUTPUT_PATH.mkdir(exist_ok=True)

MARVEL_CLEAN_PATH = OUTPUT_PATH / 'PersonaggiMarvelFive_clean.csv'
marvel_characters.to_csv(MARVEL_CLEAN_PATH, index=False)

print(f'Dataset Marvel salvato in: {MARVEL_CLEAN_PATH}')

Dataset Marvel salvato in: /Users/marcogreco/ComicGraph/Dataset/Puliti/PersonaggiMarvelFive_clean.csv


In [5]:
# Analisi dei valori Unknown e dei mancanti residui per colonna
unknown_count = (
    marvel_characters.eq('Unknown')
    .sum()
    .rename('valori_Unknown')
)

missing_count = (
    marvel_characters.isna()
    .sum()
    .rename('valori_mancanti_NaN')
)

value_quality_report = pd.concat([unknown_count, missing_count], axis=1)
value_quality_report['totale_vuoti_o_unknown'] = (
    value_quality_report['valori_Unknown']
    + value_quality_report['valori_mancanti_NaN']
)

value_quality_report.sort_values(
    'totale_vuoti_o_unknown',
    ascending=False
)

,valori_Unknown,valori_mancanti_NaN,totale_vuoti_o_unknown
EYE,9764,0,9764
HAIR,4261,0,4261
ID,3767,0,3767
ALIGN,2809,0,2809
APPEARANCES,0,1093,1093
SEX,851,0,851
Year First Appearance,0,812,812
page_id,0,0,0
name,0,0,0
urlslug,0,0,0


# Pulizia del dataset personaggi DC

## Operazioni eseguite sul dataset DC

1. **Cancellazione della colonna `GSM`** perché contiene quasi esclusivamente valori mancanti.
2. **Cancellazione della colonna `FIRST APPEARANCE`** perché duplicava l'informazione contenuta nell'anno numerico.
3. **Rinomina della colonna `YEAR`** in `Year First Appearance` per rendere più chiaro il significato del campo.
4. **Gestione dei mancanti in `EYE` e `HAIR`**: i valori vuoti sono stati sostituiti con `Unknown`, senza inventare caratteristiche del personaggio.
5. **Gestione dei tre valori mancanti in `ALIVE`**: Hatshephut II è stata impostata come `Deceased Characters`, mentre Leander e Mrblonde267/Buddy Blank sono stati impostati come `Living Characters`.
6. **Mantenimento dei mancanti numerici**: i valori vuoti di `APPEARANCES` e `Year First Appearance` restano `NaN`, perché non possono essere dedotti in modo affidabile.

I valori mancanti di `ID`, `ALIGN` e `SEX` non sono ancora stati modificati.

In [6]:
# Caricamento del dataset DC 
DC_CHARACTERS = pd.read_csv(DC_CHARACTERS_PATH)

DC_CHARACTERS = DC_CHARACTERS.drop(
    columns=['GSM', 'FIRST APPEARANCE']
)
DC_CHARACTERS = DC_CHARACTERS.rename(
    columns={'YEAR': 'Year First Appearance'}
)

print('Dimensioni:', DC_CHARACTERS.shape)
print('Colonne:', list(DC_CHARACTERS.columns))

missing_dc = (
    DC_CHARACTERS.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame('valori_mancanti')
)
missing_dc['percentuale'] = (
    missing_dc['valori_mancanti'] / len(DC_CHARACTERS) * 100
).round(2)

missing_dc[missing_dc['valori_mancanti'] > 0]

Dimensioni: (6896, 11)
Colonne: ['page_id', 'name', 'urlslug', 'ID', 'ALIGN', 'EYE', 'HAIR', 'SEX', 'ALIVE', 'APPEARANCES', 'Year First Appearance']


,valori_mancanti,percentuale
EYE,3628,52.61
HAIR,2274,32.98
ID,2013,29.19
ALIGN,601,8.72
APPEARANCES,355,5.15
SEX,125,1.81
Year First Appearance,69,1.00
ALIVE,3,0.04


In [7]:
# Gestione dei valori mancanti DC
DC_CHARACTERS[['EYE', 'HAIR']] = (
    DC_CHARACTERS[['EYE', 'HAIR']].fillna('Unknown')
)

# Assegna ALIVE ai tre record mancanti
alive_values_dc = {
    'Hatshephut II (New Earth)': 'Deceased Characters',
    'Leander (New Earth)': 'Living Characters',
    'Mrblonde267\\/Buddy Blank (New Earth)': 'Living Characters',
}

for character_name, alive_status in alive_values_dc.items():
    DC_CHARACTERS.loc[
        DC_CHARACTERS['name'] == character_name,
        'ALIVE'
    ] = alive_status

print('Valori ALIVE assegnati:')
print(
    DC_CHARACTERS.loc[
        DC_CHARACTERS['name'].isin(alive_values_dc),
        ['name', 'ALIVE']
    ].to_string(index=False)
)

# I mancanti numerici restano NaN
DC_CHARACTERS['APPEARANCES'] = pd.to_numeric(
    DC_CHARACTERS['APPEARANCES'], errors='coerce'
)
DC_CHARACTERS['Year First Appearance'] = pd.to_numeric(
    DC_CHARACTERS['Year First Appearance'], errors='coerce'
)

remaining_missing_dc = (
    DC_CHARACTERS.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame('valori_mancanti')
)
remaining_missing_dc[remaining_missing_dc['valori_mancanti'] > 0]

Valori ALIVE assegnati:
                                name               ALIVE
           Hatshephut II (New Earth) Deceased Characters
                 Leander (New Earth)   Living Characters
Mrblonde267\/Buddy Blank (New Earth)   Living Characters


,valori_mancanti
ID,2013
ALIGN,601
APPEARANCES,355
SEX,125
Year First Appearance,69


In [8]:
# Correzione anomalia: scambio dei campi per i personaggi con 'Auburn Hair' nella colonna EYE
mask_auburn = DC_CHARACTERS['EYE'] == 'Auburn Hair'

# 1. Impostiamo 'Auburn' nei capelli
DC_CHARACTERS.loc[mask_auburn, 'HAIR'] = 'Auburn'

# 2. Impostiamo 'Unknown' negli occhi
DC_CHARACTERS.loc[mask_auburn, 'EYE'] = 'Unknown'

print(f"Righe corrette: {mask_auburn.sum()}")
print(DC_CHARACTERS.loc[mask_auburn, ['name', 'EYE', 'HAIR']])


Righe corrette: 7
                                 name      EYE    HAIR
872           Marcia King (New Earth)  Unknown  Auburn
1776  Anthony Angelo, Jr. (New Earth)  Unknown  Auburn
3600                Tawna (New Earth)  Unknown  Auburn
3623             Razerkut (New Earth)  Unknown  Auburn
3627         Carla Draper (New Earth)  Unknown  Auburn
4070      Madolyn Corbett (New Earth)  Unknown  Auburn
5346         Sally Milton (New Earth)  Unknown  Auburn


In [9]:
# Salvataggio del dataset DC modificato
OUTPUT_PATH = BASE_PATH / 'Dataset' / 'Puliti'
OUTPUT_PATH.mkdir(exist_ok=True)

DC_CLEAN_PATH = OUTPUT_PATH / 'PersonaggiDCFive_clean.csv'
DC_CHARACTERS.to_csv(DC_CLEAN_PATH, index=False)

print(f'Dataset DC salvato in: {DC_CLEAN_PATH}')

Dataset DC salvato in: /Users/marcogreco/ComicGraph/Dataset/Puliti/PersonaggiDCFive_clean.csv
